In [2]:
import json
import os
import networkx as nx
from datetime import datetime

# --- CẤU HÌNH ĐƯỜNG DẪN ---
DATA_DIR = "./dataset"
RESULTS_DIR = "./results"
os.makedirs(RESULTS_DIR, exist_ok=True)

In [3]:
# --- 1. LOAD DỮ LIỆU ---
def load_json(filepath, default_val=None):
    try:
        with open(filepath, 'r', encoding='utf-8') as f:
            return json.load(f)
    except FileNotFoundError:
        return default_val

def load_jsonl(filepath):
    data = []
    try:
        with open(filepath, 'r', encoding='utf-8') as f:
            for line in f:
                if line.strip():
                    data.append(json.loads(line))
    except FileNotFoundError:
        pass
    return data

# Load các file
cluster_summary = load_json(os.path.join(RESULTS_DIR, 'cluster_summary.json'), {"clusters": []})
services_data = load_json(os.path.join(DATA_DIR, 'services.json'), {"services": [], "edges": []})
alerts = load_jsonl(os.path.join(DATA_DIR, 'alerts_sample.jsonl'))
history_data = load_json(os.path.join(DATA_DIR, 'incidents_history.json'), {"incidents": []})
history = history_data.get("incidents", [])

In [4]:
# --- 2. BUILD SERVICE GRAPH ---
G = nx.DiGraph()

# Add nodes
for svc in services_data.get("services", []):
    G.add_node(svc.get("name"))
for store in services_data.get("stores", []):
    G.add_node(store.get("name"))

# Add edges
for edge in services_data.get("edges", []):
    # Chiều mũi tên: Caller -> Callee (A gọi B)
    G.add_edge(edge["from"], edge["to"])

In [5]:
# --- 3. GRAPH & TEMPORAL SCORER ---
def score_candidates(cluster_alerts, graph):
    if not cluster_alerts:
        return []
    
    # Lấy danh sách service đang alert
    alert_services = list(set([a["service"] for a in cluster_alerts]))
    
    # Tạo subgraph và reverse để chạy PageRank (Service bị gọi nhiều sẽ có điểm cao)
    subgraph = graph.subgraph(alert_services)
    try:
        R = subgraph.reverse(copy=True)
        pr_scores = nx.pagerank(R, alpha=0.85)
    except:
        pr_scores = {s: 1.0 for s in alert_services} # Fallback
        
    # Chuẩn hóa PageRank
    max_pr = max(pr_scores.values()) if pr_scores else 1
    pr_norm = {k: v/max_pr for k, v in pr_scores.items()}
    
    # Xử lý Temporal Score (Convert string timestamp -> float)
    for a in cluster_alerts:
        a["_ts_float"] = datetime.strptime(a["ts"], "%Y-%m-%dT%H:%M:%SZ").timestamp()
        
    cluster_alerts.sort(key=lambda x: x["_ts_float"])
    first_time = cluster_alerts[0]["_ts_float"]
    last_time = cluster_alerts[-1]["_ts_float"]
    
    final_scores = {}
    for svc in alert_services:
        # Tìm alert đầu tiên của service này
        svc_first_time = next(a["_ts_float"] for a in cluster_alerts if a["service"] == svc)
        
        # Scale temporal score (0.0 -> 1.0)
        if last_time == first_time:
            t_score = 1.0
        else:
            t_score = 1.0 - (svc_first_time - first_time) / (last_time - first_time)
        
        # Công thức: 0.6 Graph + 0.4 Temporal
        final_scores[svc] = 0.6 * pr_norm.get(svc, 0) + 0.4 * t_score
        
    # Sort điểm giảm dần
    return sorted(final_scores.items(), key=lambda x: x[1], reverse=True)

In [6]:
# --- 4. RETRIEVAL SCORER (kNN-style) ---
def retrieve_similar_incidents(target_service, alert_services, history_list, top_k=3):
    scored_history = []
    for inc in history_list:
        score = 0.0
        
        # +0.4 nếu root_cause_service trùng
        if inc.get("root_cause_service") == target_service:
            score += 0.4
        
        # +0.2 cho mỗi service overlap
        history_svcs = set(inc.get("services_involved", []))
        overlap = set(alert_services).intersection(history_svcs)
        score += min(0.4, 0.2 * len(overlap))
        
        if score >= 0.2:
            scored_history.append((inc, score))
            
    scored_history.sort(key=lambda x: x[1], reverse=True)
    return [item[0] for item in scored_history[:top_k]]

In [7]:
# --- 5. MAIN PIPELINE ---
results_output = {
    "clusters_analyzed": len(cluster_summary.get("clusters", [])),
    "results": []
}

for cluster in cluster_summary.get("clusters", []):
    c_id = cluster["cluster_id"]
    c_fingerprints = cluster.get("fingerprints", [])
    
    # Map raw alerts vào cluster thông qua fingerprint (vì alert gốc không có cluster_id)
    c_alerts = []
    for a in alerts:
        fp = f"{a['service']}|{a['metric']}|{a['severity']}"
        if fp in c_fingerprints:
            c_alerts.append(a)
            
    if not c_alerts:
        continue
        
    # Chấm điểm candidates
    candidates = score_candidates(c_alerts, G)
    top1_svc = candidates[0][0] if candidates else "unknown"
    top1_score = candidates[0][1] if candidates else 0.0
    
    # Kéo history
    alert_svcs = [c[0] for c in candidates]
    similar_incidents = retrieve_similar_incidents(top1_svc, alert_svcs, history, top_k=3)
    
    # Phân loại Root Cause & Lấy Action
    if similar_incidents:
        best_match = similar_incidents[0]
        rc_class = best_match.get("root_cause_class", "other")
        # Chuyển remediation string thành mảng theo yêu cầu output
        actions = [best_match.get("remediation", "Investigate manually")]
        sim_ids = [inc.get("id") for inc in similar_incidents]
    else:
        rc_class = "other"
        actions = ["Investigate manually - No similar history found"]
        sim_ids = []
        
    record = {
        "cluster_id": c_id,
        "graph_top3": [[c[0], round(c[1], 2)] for c in candidates[:3]],
        "root_cause": top1_svc,
        "class": rc_class,
        "confidence": round(top1_score, 2),
        "actions": actions,
        "reasoning": f"Service '{top1_svc}' có điểm cao nhất dựa trên đồ thị (nhiều downstream phụ thuộc) và xuất hiện cảnh báo sớm nhất trong chuỗi.",
        "similar_incidents": sim_ids,
        "method": "graph+retrieval"
    }
    results_output["results"].append(record)

# --- 6. GHI FILE ---
output_path = os.path.join(RESULTS_DIR, 'rca_output.json')
with open(output_path, 'w', encoding='utf-8') as f:
    json.dump(results_output, f, indent=2, ensure_ascii=False)

print(f"Đã phân tích xong {len(results_output['results'])} clusters.")
print(f"Kết quả được ghi tại: {output_path}")

Đã phân tích xong 5 clusters.
Kết quả được ghi tại: ./results\rca_output.json
